# EuroSAT RGB Classification and Image Retrieval

Final training, evaluation, artifact export, and Hugging Face deployment
notebook.

The notebook uses one configuration value, `MODEL_NAME`, for model training,
saved filenames, metric reports, and deployment files.


In [ ]:
import gc
import json
import random
import re
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT_DIR = Path("/kaggle/input/datasets/glitchr/eurodata")
RGB_DIR = ROOT_DIR / "EuroSAT_RGB"
WORK_DIR = Path("/kaggle/working/eurodata_optical")
MODEL_DIR = WORK_DIR / "models"
REPORT_DIR = WORK_DIR / "reports"
DEPLOY_DIR = WORK_DIR / "huggingface_space"

for directory in (WORK_DIR, MODEL_DIR, REPORT_DIR, DEPLOY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "DenseNet121"
IMAGE_SIZE = 64
IMAGE_CHANNELS = 3
INPUT_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, IMAGE_CHANNELS)
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 1e-3
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]
CLASS_TO_INDEX = {
    label: index for index, label in enumerate(CLASS_NAMES)
}
NUM_CLASSES = len(CLASS_NAMES)
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

print("TensorFlow:", tf.__version__)
print("Model:", MODEL_NAME)
print("Classes:", NUM_CLASSES)


## 1. Build and validate the dataset index

In [ ]:
def extract_file_id(image_path):
    numbers = re.findall(r"\d+", image_path.stem)
    return int(numbers[-1]) if numbers else None


def collect_rgb_files(root_dir):
    records = []

    for label in CLASS_NAMES:
        class_dir = root_dir / label

        if not class_dir.exists():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")

        for image_path in sorted(class_dir.rglob("*")):
            if (
                image_path.is_file()
                and image_path.suffix.lower() in VALID_EXTENSIONS
            ):
                records.append(
                    {
                        "file_id": extract_file_id(image_path),
                        "label": label,
                        "target": CLASS_TO_INDEX[label],
                        "image_path": str(image_path),
                    }
                )

    dataframe = pd.DataFrame(records)
    dataframe = dataframe.dropna(
        subset=["file_id", "label", "image_path"]
    )
    dataframe["file_id"] = dataframe["file_id"].astype(int)

    duplicate_mask = dataframe.duplicated(
        subset=["label", "file_id"],
        keep=False,
    )
    duplicate_count = int(duplicate_mask.sum())

    dataframe = dataframe.drop_duplicates(
        subset=["label", "file_id"],
        keep="first",
    ).reset_index(drop=True)

    print("Rows after deduplication:", len(dataframe))
    print("Duplicate rows removed:", duplicate_count)
    return dataframe


final_df = collect_rgb_files(RGB_DIR)

assert final_df["image_path"].isna().sum() == 0
assert final_df["label"].isin(CLASS_NAMES).all()
assert final_df["target"].between(0, NUM_CLASSES - 1).all()
assert not final_df.duplicated(["label", "file_id"]).any()

display(final_df.head())
display(final_df["label"].value_counts().sort_index())


In [ ]:
train_df, temporary_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=SEED,
    stratify=final_df["label"],
)

val_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "images": [len(train_df), len(val_df), len(test_df)],
    }
)

display(split_summary)


## 2. TensorFlow input pipeline

In [ ]:
def decode_image(image_path, target):
    image_bytes = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(
        image_bytes,
        channels=IMAGE_CHANNELS,
    )
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    image = tf.cast(image, tf.float32)
    target = tf.one_hot(target, depth=NUM_CLASSES)
    return image, target


def augment_image(image, target):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=12.0)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, target


def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    targets = dataframe["target"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, targets))

    if training:
        dataset = dataset.shuffle(
            len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        decode_image,
        num_parallel_calls=AUTOTUNE,
    )

    if training:
        dataset = dataset.map(
            augment_image,
            num_parallel_calls=AUTOTUNE,
        )

    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)


## 3. Build and train the model

In [ ]:
MODEL_REGISTRY = {
    "DenseNet121": tf.keras.applications.DenseNet121,
    "ResNet50": tf.keras.applications.ResNet50,
    "MobileNetV2": tf.keras.applications.MobileNetV2,
    "EfficientNetB0": tf.keras.applications.EfficientNetB0,
}

PREPROCESS_REGISTRY = {
    "DenseNet121": tf.keras.applications.densenet.preprocess_input,
    "ResNet50": tf.keras.applications.resnet.preprocess_input,
    "MobileNetV2": tf.keras.applications.mobilenet_v2.preprocess_input,
    "EfficientNetB0": tf.keras.applications.efficientnet.preprocess_input,
}


def build_model(model_name):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(
            f"Unsupported model: {model_name}. "
            f"Choose from {list(MODEL_REGISTRY)}"
        )

    inputs = tf.keras.Input(
        shape=INPUT_SHAPE,
        name="optical_image",
    )
    preprocess = PREPROCESS_REGISTRY[model_name]
    x = tf.keras.layers.Lambda(
        preprocess,
        name="model_preprocessing",
    )(inputs)

    backbone = MODEL_REGISTRY[model_name](
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE,
        pooling="avg",
    )
    backbone.trainable = False

    x = backbone(x, training=False)
    x = tf.keras.layers.BatchNormalization(
        name="embedding_batch_norm"
    )(x)
    x = tf.keras.layers.Dense(
        256,
        activation="relu",
        name="embedding",
    )(x)
    x = tf.keras.layers.Dropout(0.30)(x)

    outputs = tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        name="classification",
    )(x)

    model = tf.keras.Model(
        inputs,
        outputs,
        name=f"{model_name}_EuroSAT_RGB",
    )
    return model


model = build_model(MODEL_NAME)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=5,
            name="top_5_accuracy",
        ),
    ],
)
model.summary()


In [ ]:
CLASSIFIER_PATH = MODEL_DIR / f"{MODEL_NAME}_classifier.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CLASSIFIER_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

training_start = time.perf_counter()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)
training_seconds = time.perf_counter() - training_start

model = tf.keras.models.load_model(
    CLASSIFIER_PATH,
    compile=False,
    safe_mode=False,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=5,
            name="top_5_accuracy",
        ),
    ],
)

test_metrics = model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1,
)

test_metrics["training_seconds"] = training_seconds
display(pd.DataFrame([test_metrics]))


## 4. Classification validation

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(REPORT_DIR / "training_history.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Training")
plt.plot(history_df["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"{MODEL_NAME} accuracy")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training")
plt.plot(history_df["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{MODEL_NAME} loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
probabilities = model.predict(test_ds, verbose=1)
predicted_classes = probabilities.argmax(axis=1)
true_classes = test_df["target"].to_numpy()

report = classification_report(
    true_classes,
    predicted_classes,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(
    REPORT_DIR / "classification_report.csv",
    index=True,
)
display(report_df)

matrix = confusion_matrix(true_classes, predicted_classes)
ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=CLASS_NAMES,
).plot(xticks_rotation=45)

plt.title(f"{MODEL_NAME} confusion matrix")
plt.tight_layout()
plt.show()


## 5. Embeddings and retrieval evaluation

Relevance definition: a retrieved gallery image is relevant when it has the
same EuroSAT class as the query. During test-set evaluation, the query image
itself is excluded from the gallery ranking.

The implementation validates Precision@K, Recall@K, F1@K, mAP, MRR, and
latency over every test query.


In [ ]:
embedding_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("embedding").output,
    name=f"{MODEL_NAME}_embedding_model",
)

EMBEDDING_MODEL_PATH = (
    MODEL_DIR / f"{MODEL_NAME}_embedding.keras"
)
embedding_model.save(EMBEDDING_MODEL_PATH)

gallery_embeddings = embedding_model.predict(
    test_ds,
    verbose=1,
).astype(np.float32)

gallery_embeddings /= np.clip(
    np.linalg.norm(gallery_embeddings, axis=1, keepdims=True),
    1e-8,
    None,
)

assert len(gallery_embeddings) == len(test_df)
assert np.isfinite(gallery_embeddings).all()
assert np.allclose(
    np.linalg.norm(gallery_embeddings, axis=1),
    1.0,
    atol=1e-5,
)

print("Embedding shape:", gallery_embeddings.shape)


In [ ]:
def evaluate_retrieval(
    dataframe,
    embeddings,
    k_values=(1, 5, 10, 20, 50),
    label_column="label",
):
    labels = dataframe[label_column].to_numpy()
    embeddings = np.asarray(embeddings, dtype=np.float32)

    if len(dataframe) != len(embeddings):
        raise ValueError(
            "Dataframe and embedding counts do not match."
        )

    if len(dataframe) < 2:
        raise ValueError("At least two images are required.")

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-8, None)

    gallery_size = len(dataframe) - 1
    valid_k_values = sorted(
        {
            min(int(k), gallery_size)
            for k in k_values
            if int(k) > 0
        }
    )

    query_rows = []
    retrieval_times = []

    for query_index in tqdm(
        range(len(dataframe)),
        desc="Evaluating all test queries",
    ):
        start = time.perf_counter()

        similarities = embeddings @ embeddings[query_index]
        similarities[query_index] = -np.inf

        ranked_indices = np.argsort(similarities)[::-1]
        ranked_indices = ranked_indices[
            ranked_indices != query_index
        ]

        retrieval_times.append(time.perf_counter() - start)

        query_label = labels[query_index]
        relevance = (
            labels[ranked_indices] == query_label
        ).astype(np.int32)

        total_relevant = int(
            np.sum(labels == query_label) - 1
        )
        cumulative_relevant = np.cumsum(relevance)

        row = {
            "query_index": query_index,
            "query_label": query_label,
            "total_relevant": total_relevant,
        }

        for k in valid_k_values:
            relevant_at_k = int(cumulative_relevant[k - 1])
            precision = relevant_at_k / k
            recall = (
                relevant_at_k / total_relevant
                if total_relevant > 0
                else 0.0
            )
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall > 0
                else 0.0
            )

            row[f"relevant_at_{k}"] = relevant_at_k
            row[f"precision_at_{k}"] = precision
            row[f"recall_at_{k}"] = recall
            row[f"f1_at_{k}"] = f1

        if total_relevant > 0:
            ranks = np.arange(1, len(relevance) + 1)
            precision_by_rank = cumulative_relevant / ranks
            average_precision = np.sum(
                precision_by_rank * relevance
            ) / total_relevant

            relevant_positions = np.flatnonzero(relevance)
            reciprocal_rank = (
                1.0 / (relevant_positions[0] + 1)
                if len(relevant_positions)
                else 0.0
            )
        else:
            average_precision = 0.0
            reciprocal_rank = 0.0

        row["average_precision"] = float(average_precision)
        row["reciprocal_rank"] = float(reciprocal_rank)
        query_rows.append(row)

    query_metrics = pd.DataFrame(query_rows)

    summary = {
        "model_name": MODEL_NAME,
        "number_of_queries": len(dataframe),
        "gallery_size_per_query": gallery_size,
        "mean_average_precision": (
            query_metrics["average_precision"].mean()
        ),
        "mean_reciprocal_rank": (
            query_metrics["reciprocal_rank"].mean()
        ),
        "average_retrieval_time_ms": (
            np.mean(retrieval_times) * 1000
        ),
        "queries_per_second": (
            1.0 / np.mean(retrieval_times)
        ),
    }

    for k in valid_k_values:
        for metric in ("precision", "recall", "f1"):
            column = f"{metric}_at_{k}"
            summary[column] = query_metrics[column].mean()

    summary_df = pd.DataFrame([summary])

    metric_columns = [
        column
        for column in query_metrics.columns
        if column.startswith(("precision_at_", "recall_at_", "f1_at_"))
    ]

    metric_values = query_metrics[metric_columns].to_numpy()
    if not np.isfinite(metric_values).all():
        raise ValueError("Retrieval metrics contain NaN or infinity.")

    if not ((metric_values >= 0) & (metric_values <= 1)).all():
        raise ValueError("Retrieval metrics must be within [0, 1].")

    return query_metrics, summary_df, valid_k_values


In [ ]:
K_VALUES = [1, 2, 3, 5, 10, 15, 20, 30, 50]

query_metrics_df, retrieval_summary_df, valid_k_values = (
    evaluate_retrieval(
        dataframe=test_df,
        embeddings=gallery_embeddings,
        k_values=K_VALUES,
    )
)

query_metrics_df.to_csv(
    REPORT_DIR / "query_retrieval_metrics.csv",
    index=False,
)
retrieval_summary_df.to_csv(
    REPORT_DIR / "retrieval_summary.csv",
    index=False,
)

display(retrieval_summary_df.T)


In [ ]:
precision_values = [
    retrieval_summary_df[f"precision_at_{k}"].iloc[0]
    for k in valid_k_values
]
recall_values = [
    retrieval_summary_df[f"recall_at_{k}"].iloc[0]
    for k in valid_k_values
]
f1_values = [
    retrieval_summary_df[f"f1_at_{k}"].iloc[0]
    for k in valid_k_values
]

plt.figure(figsize=(9, 6))
plt.plot(
    valid_k_values,
    precision_values,
    marker="o",
    label="Mean Precision@K",
)
plt.plot(
    valid_k_values,
    recall_values,
    marker="x",
    label="Mean Recall@K",
)
plt.plot(
    valid_k_values,
    f1_values,
    marker="*",
    label="Mean F1@K",
)

plt.xlabel("Top-K retrieved images")
plt.ylabel("Mean score")
plt.title(f"{MODEL_NAME}: complete test-set retrieval")
plt.xticks(valid_k_values)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
class_metric_columns = [
    "precision_at_5",
    "recall_at_5",
    "f1_at_5",
    "precision_at_10",
    "recall_at_10",
    "f1_at_10",
]

class_metrics_df = (
    query_metrics_df.groupby("query_label")[class_metric_columns]
    .mean()
    .reset_index()
)

class_metrics_df.to_csv(
    REPORT_DIR / "class_retrieval_metrics.csv",
    index=False,
)
display(class_metrics_df.round(4))


## 6. Retrieval demonstration

In [ ]:
def retrieve_by_index(
    query_index,
    dataframe,
    embeddings,
    top_k=10,
):
    similarities = embeddings @ embeddings[query_index]
    similarities[query_index] = -np.inf
    ranked_indices = np.argsort(similarities)[::-1][:top_k]

    query_label = dataframe.iloc[query_index]["label"]
    rows = []

    for rank, gallery_index in enumerate(
        ranked_indices,
        start=1,
    ):
        row = dataframe.iloc[gallery_index]
        rows.append(
            {
                "rank": rank,
                "gallery_index": int(gallery_index),
                "file_id": int(row["file_id"]),
                "label": row["label"],
                "image_path": row["image_path"],
                "similarity": float(
                    similarities[gallery_index]
                ),
                "relevant": int(row["label"] == query_label),
            }
        )

    return pd.DataFrame(rows)


def display_retrieval(query_index, results, dataframe):
    query_row = dataframe.iloc[query_index]
    total_images = len(results) + 1
    columns = 4
    rows = int(np.ceil(total_images / columns))

    plt.figure(figsize=(4 * columns, 4 * rows))

    plt.subplot(rows, columns, 1)
    plt.imshow(Image.open(query_row["image_path"]).convert("RGB"))
    plt.title(f"Query\n{query_row['label']}")
    plt.axis("off")

    for position, result in results.iterrows():
        plt.subplot(rows, columns, position + 2)
        plt.imshow(
            Image.open(result["image_path"]).convert("RGB")
        )
        plt.title(
            f"Rank {int(result['rank'])}\n"
            f"{result['label']}\n"
            f"{result['similarity']:.3f}"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


query_index = 0
retrieval_results = retrieve_by_index(
    query_index,
    test_df,
    gallery_embeddings,
    top_k=10,
)

display(retrieval_results)
display_retrieval(query_index, retrieval_results, test_df)


## 7. Export a complete Hugging Face Space

This section creates a self-contained deployment directory containing the
classifier, embedding model, normalized gallery embeddings, gallery metadata,
gallery images, `app.py`, `config.json`, `requirements.txt`, and `README.md`.

The same `MODEL_NAME` value is used in every generated artifact name.


In [ ]:
CLASSIFIER_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_classifier.keras"
)
EMBEDDING_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_embedding.keras"
)
EMBEDDINGS_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_gallery_embeddings.npy"
)
METADATA_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_gallery_metadata.csv"
)
GALLERY_DIR = DEPLOY_DIR / "gallery"
GALLERY_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(CLASSIFIER_PATH, CLASSIFIER_DEPLOY_PATH)
shutil.copy2(EMBEDDING_MODEL_PATH, EMBEDDING_DEPLOY_PATH)
np.save(EMBEDDINGS_DEPLOY_PATH, gallery_embeddings)

deployment_metadata = test_df[
    ["file_id", "label", "image_path"]
].copy()

relative_paths = []

for row in tqdm(
    deployment_metadata.itertuples(index=False),
    total=len(deployment_metadata),
    desc="Copying deployment gallery",
):
    source = Path(row.image_path)
    class_dir = GALLERY_DIR / row.label
    class_dir.mkdir(parents=True, exist_ok=True)

    destination = class_dir / source.name

    if not destination.exists():
        shutil.copy2(source, destination)

    relative_paths.append(
        destination.relative_to(DEPLOY_DIR).as_posix()
    )

deployment_metadata["relative_image_path"] = relative_paths
deployment_metadata = deployment_metadata.drop(
    columns=["image_path"]
)
deployment_metadata.to_csv(
    METADATA_DEPLOY_PATH,
    index=False,
)

config = {
    "space_title": "EuroSAT Optical Retrieval",
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "image_channels": IMAGE_CHANNELS,
    "class_names": CLASS_NAMES,
    "model_file": CLASSIFIER_DEPLOY_PATH.name,
    "embedding_model_file": EMBEDDING_DEPLOY_PATH.name,
    "gallery_embeddings_file": EMBEDDINGS_DEPLOY_PATH.name,
    "gallery_metadata_file": METADATA_DEPLOY_PATH.name,
    "top_k_default": 5,
    "distance": "cosine_similarity",
}

with open(
    DEPLOY_DIR / "config.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(config, file, indent=2)

print("Deployment metadata:", deployment_metadata.shape)


In [ ]:
APP_CODE = 'import json\nfrom pathlib import Path\n\nimport gradio as gr\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nfrom PIL import Image\n\nROOT = Path(__file__).resolve().parent\n\nwith open(ROOT / "config.json", "r", encoding="utf-8") as file:\n    CONFIG = json.load(file)\n\nMODEL_NAME = CONFIG["model_name"]\nIMAGE_SIZE = int(CONFIG["image_size"])\nCLASS_NAMES = CONFIG["class_names"]\nTOP_K_DEFAULT = int(CONFIG.get("top_k_default", 5))\n\nMODEL_PATH = ROOT / CONFIG["model_file"]\nEMBEDDING_MODEL_PATH = ROOT / CONFIG["embedding_model_file"]\nEMBEDDINGS_PATH = ROOT / CONFIG["gallery_embeddings_file"]\nMETADATA_PATH = ROOT / CONFIG["gallery_metadata_file"]\n\nclassifier = tf.keras.models.load_model(\n    MODEL_PATH,\n    compile=False,\n    safe_mode=False,\n)\nembedding_model = tf.keras.models.load_model(\n    EMBEDDING_MODEL_PATH,\n    compile=False,\n    safe_mode=False,\n)\n\ngallery_embeddings = np.load(EMBEDDINGS_PATH).astype(np.float32)\ngallery_metadata = pd.read_csv(METADATA_PATH)\n\nnorms = np.linalg.norm(gallery_embeddings, axis=1, keepdims=True)\ngallery_embeddings = gallery_embeddings / np.clip(norms, 1e-8, None)\n\n\ndef prepare_image(image):\n    image = image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE))\n    array = np.asarray(image, dtype=np.float32)\n    return np.expand_dims(array, axis=0)\n\n\ndef predict_class(image):\n    batch = prepare_image(image)\n    probabilities = classifier.predict(batch, verbose=0)[0]\n    return {\n        CLASS_NAMES[index]: float(probabilities[index])\n        for index in np.argsort(probabilities)[::-1][:5]\n    }\n\n\ndef retrieve_images(image, top_k):\n    batch = prepare_image(image)\n    query_embedding = embedding_model.predict(batch, verbose=0)\n    query_embedding = query_embedding / np.clip(\n        np.linalg.norm(query_embedding, axis=1, keepdims=True),\n        1e-8,\n        None,\n    )\n\n    similarities = gallery_embeddings @ query_embedding[0]\n    top_k = min(int(top_k), len(gallery_metadata))\n    ranked_indices = np.argsort(similarities)[::-1][:top_k]\n\n    gallery = []\n    rows = []\n\n    for rank, index in enumerate(ranked_indices, start=1):\n        row = gallery_metadata.iloc[index]\n        image_path = ROOT / row["relative_image_path"]\n\n        if image_path.exists():\n            caption = (\n                f"Rank {rank} | {row[\'label\']} | "\n                f"similarity={similarities[index]:.4f}"\n            )\n            gallery.append((str(image_path), caption))\n\n        rows.append(\n            [\n                rank,\n                row["label"],\n                int(row["file_id"]),\n                float(similarities[index]),\n            ]\n        )\n\n    return gallery, rows\n\n\ndef run_inference(image, top_k):\n    if image is None:\n        return {}, [], []\n\n    predictions = predict_class(image)\n    gallery, rows = retrieve_images(image, top_k)\n    return predictions, gallery, rows\n\n\nwith gr.Blocks(title=CONFIG.get("space_title", MODEL_NAME)) as demo:\n    gr.Markdown(\n        f"# {CONFIG.get(\'space_title\', MODEL_NAME)}\\n"\n        f"Model: **{MODEL_NAME}**"\n    )\n\n    with gr.Row():\n        input_image = gr.Image(type="pil", label="Upload optical image")\n        top_k = gr.Slider(\n            minimum=1,\n            maximum=20,\n            value=TOP_K_DEFAULT,\n            step=1,\n            label="Number of retrieved images",\n        )\n\n    run_button = gr.Button("Classify and retrieve")\n\n    class_output = gr.Label(\n        num_top_classes=5,\n        label="Land-cover prediction",\n    )\n    gallery_output = gr.Gallery(\n        label="Similar images",\n        columns=5,\n        object_fit="contain",\n        height="auto",\n    )\n    table_output = gr.Dataframe(\n        headers=["rank", "label", "file_id", "similarity"],\n        datatype=["number", "str", "number", "number"],\n        interactive=False,\n        label="Retrieval results",\n    )\n\n    run_button.click(\n        fn=run_inference,\n        inputs=[input_image, top_k],\n        outputs=[class_output, gallery_output, table_output],\n    )\n\nif __name__ == "__main__":\n    demo.launch()\n'

(DEPLOY_DIR / 'app.py').write_text(APP_CODE, encoding='utf-8')
print('Saved:', DEPLOY_DIR / 'app.py')


In [ ]:
REQUIREMENTS = 'tensorflow-cpu==2.16.1\ngradio==5.35.0\nnumpy==1.26.4\npandas==2.2.2\nPillow==10.4.0\n'
README_TEXT = '---\ntitle: EuroSAT Optical Retrieval\nemoji: 🛰️\ncolorFrom: blue\ncolorTo: green\nsdk: gradio\nsdk_version: 5.35.0\napp_file: app.py\npinned: false\n---\n\n# EuroSAT Optical Retrieval\n\nThis Hugging Face Space uses **DenseNet121** for EuroSAT RGB land-cover\nclassification and same-modal image retrieval.\n\n## Required generated artifacts\n\nRun the final notebook through the deployment export section. It creates:\n\n- `DenseNet121_classifier.keras`\n- `DenseNet121_embedding.keras`\n- `DenseNet121_gallery_embeddings.npy`\n- `DenseNet121_gallery_metadata.csv`\n- `gallery/`\n- `config.json`\n- `app.py`\n- `requirements.txt`\n\nUpload the complete generated deployment directory to a Hugging Face Space.\nLarge `.keras` and `.npy` files should be tracked using Git LFS.\n\n## Retrieval definition\n\nA gallery image is relevant when its EuroSAT class label matches the query\nlabel. The query itself is excluded during same-dataset evaluation.\n\nReported metrics include Precision@K, Recall@K, F1@K, mAP, MRR, and average\nretrieval latency.\n'

(DEPLOY_DIR / 'requirements.txt').write_text(
    REQUIREMENTS,
    encoding='utf-8',
)
(DEPLOY_DIR / 'README.md').write_text(
    README_TEXT,
    encoding='utf-8',
)
print('Saved deployment support files.')


In [ ]:
required_files = [
    DEPLOY_DIR / "app.py",
    DEPLOY_DIR / "requirements.txt",
    DEPLOY_DIR / "README.md",
    DEPLOY_DIR / "config.json",
    CLASSIFIER_DEPLOY_PATH,
    EMBEDDING_DEPLOY_PATH,
    EMBEDDINGS_DEPLOY_PATH,
    METADATA_DEPLOY_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing deployment files: {missing_files}"
    )

saved_embeddings = np.load(EMBEDDINGS_DEPLOY_PATH)
saved_metadata = pd.read_csv(METADATA_DEPLOY_PATH)

assert saved_embeddings.shape[0] == len(saved_metadata)
assert saved_embeddings.shape[1] == gallery_embeddings.shape[1]
assert saved_metadata["relative_image_path"].isna().sum() == 0
assert all(
    (DEPLOY_DIR / path).exists()
    for path in saved_metadata["relative_image_path"]
)

archive_base = WORK_DIR / f"{MODEL_NAME}_huggingface_space"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=DEPLOY_DIR,
)

print("Deployment validation passed.")
print("Space directory:", DEPLOY_DIR)
print("ZIP archive:", archive_path)


## Metric interpretation

- **Precision@K**: fraction of the top K results that have the query class.
- **Recall@K**: fraction of all same-class gallery images recovered in top K.
- **F1@K**: harmonic mean of Precision@K and Recall@K.
- **mAP**: mean Average Precision over the complete ranking for every query.
- **MRR**: mean reciprocal rank of the first relevant result.

For a large class, Recall@5 can remain numerically small even when
Precision@5 is high because only five images can be returned.
